In [11]:
import pymupdf
import os
import time
from datetime import datetime
import numpy as np
import pandas as pd
import sys
# Add /modules/extract_text directories to where packages could be imported
sys.path.append("../../modules/extract_text")
import BlockFont as bf

# Import PDF
continent_selected = "southamerica" # africa/asia/europe/northamerica/southamerica
pdf_dir = "../../data/original_report/" + "usgs_" + continent_selected + '/' # Input directory
pdf_names = os.listdir(pdf_dir)
pdf_paths = [os.path.join(pdf_dir, pdf_name) for pdf_name in pdf_names]
txt_dir = "../../data/policy_section_txt/" + "usgs_" + continent_selected + '/'# Output directory

## Initialize a log file
log_folder = "../../log/" # Folder where all files are stored
log_name = 'extractpolicy_' + str(int(time.time())) + '.txt' # Both name of task and timestamp included
log_path = os.path.join(log_folder, log_name)

### Iterate through PDF files
with open(log_path, 'w', encoding='utf-8') as log: # IMPORTANT: Designate encoding as 'utf-8' to avoid error caused by unrecognizable symbols under 'gbk' encoding
    counter = 0 # Initialize the counter
    for i in range(len(pdf_names)):
        pdf_path = pdf_paths[i]
        pdf = pymupdf.open(pdf_path)

        # --------------Start the timer and counter--------------
        counter += 1
        start_time = time.perf_counter(); start_time_legible = str(datetime.now())
        country = pdf_names[i].split('.')[0].split('_')[0]; year = pdf_names[i].split('.')[0].split('_')[1]
        message_start = f"PDF loaded. Processing {counter}/{len(pdf_names)} [{country} in {year} from: {pdf_path}]"
        print(message_start)
        log.write(start_time_legible + '\n')
        log.write(message_start + '\n\n') # Add two line breaks
        # -------------------------------------------------------

        # 0. Retain only pages in the main body of the file (remove covers and appendices)
        start_page_num = 0
        end_page_num = len(pdf) # IMPORTANT: Variable 'end_page_num' records the page number after the last page. But since range() function exactly excludes the redundant page when generating a sequence, this index well fits our end.
        for page in pdf:
            page_text = page.get_text('text')
            if page_text.count("The Mineral Industry of") != 0: # Case-sensitive, where the main body starts
                start_page_num = page.number
            # CRITICAL DEBUG: Previous discriminant "page_text.count("TABLE") != 0" could fail as in some reports appendices turn out start with "tABLe" or other bizarre composition of cases. But directly search for case-insensitive 'table' could mistake reference to appendices in the main text, so we try to add another condition, detecting the joint occurrence of two phrases.
            if len(page.search_for('table')) !=0 and len(page.search_for(': production of mineral commodities')) !=0: # Case-insensitive, the page after where the main body ends
                end_page_num = page.number
                break
        # CRITICAL DEBUG: Skip the current PDF file if such issue is detected
        if start_page_num == end_page_num:
            # ---------------Provisional Termination-----------------
            end_time = time.perf_counter()
            end_time_legible = str(datetime.now())
            execution_time = end_time - start_time # Calculate execution interval
            message_end = f"Bizarre format! Annexed table is embedded in the 1st page of the main body. Execution time {execution_time:.2f} seconds"
            print(message_end)
            log.write(end_time_legible + '\n')
            log.write(message_end + '\n\n') # Add two line breaks
            continue
            # --------------------------------------------------------
        body_range = list(range(start_page_num, end_page_num))
        pdf.select(body_range) # IMPORTANT: Different from ordinary python commands which make a copy of the original object, this command directly amends the object "pdf". So DO NOT write "pdf = pdf.select(...)".


        ### 1. Iterate through pages within the PDF, remove headers, footers and footnotes; then create a dataframe storing all the layout properties of blocks identified by .get_text("blocks/dict") and sort them in accordance with normal reading sequence (by column, top-down in each column)
        is_firstpage = True # Indicator for whether it's first page, correspondingly whether to create or concatenate with the aggregate dataframe
        allpages_mainbody_wizref_properties_df = pd.DataFrame()
        for j in range(len(pdf)): # Iterate through pages in the PDF file
            ## 1.0 [Input] Extract all blocks on the current page
            blocks_dict_list = pdf[j].get_text("dict")['blocks'] # Extract plain text in blocks on the page
            blocks_list = pdf[j].get_text("blocks") # Extract detailed format info of blocks on the page

            ## 1.1 [Property Extraction] Retrieve properties of all blocks from encapsulated function "block_format([BLOCK_DICT])" from /modules/extract_text/BlockFont.py
            all_blocks_properties_df = pd.DataFrame([bf.block_format(block_dict) for block_dict in blocks_dict_list])
            all_blocks_properties_df['page_in_body'] = j # Add a column in block property matrix to record the current page number (IMPORTANT: It's index of the page after removing covers and appendices, not the page number in original PDF file)
            # Only retain blocks belonging to main-text and reference categories
            mainbody_wizref_properties_df = all_blocks_properties_df.query("not (is_header or is_footer or is_footnote)")

            ## 1.2 [Central Axis Location] Figure out the x-coordinate for central axis, based on which the page is split into two columns
            # Calculate the x-coordinate for left boundary of the right column and vice versa. 
            rightcolumn_start_x_coordinate = min([x_coord for x_coord in mainbody_wizref_properties_df['upperleft_x_coordinate'] if x_coord > 300])
            leftcolumn_end_x_coordinate = max([x_coord for x_coord in mainbody_wizref_properties_df['lowerright_x_coordinate'] if x_coord < rightcolumn_start_x_coordinate])
            # Central axis is set as threading through the midpoint of these two coordinates. 
            central_axis_x_coordinate = (leftcolumn_end_x_coordinate + rightcolumn_start_x_coordinate)/2

            ## 1.3 [Blocks grouping & sorting] Group main-text and reference blocks by column, and sort blocks in each group by y-coordinate, before rejoining them in the dataframe by row
            # Left-column blocks sorting and indexing
            leftcolumn_blocks_properties_df = mainbody_wizref_properties_df.query("upperleft_x_coordinate < @central_axis_x_coordinate") # IMPORTANT: The selected coordinate should be the same for two discriminants to avoid rare occassions where a long banner horizonally spans across the central axis.
            leftcolumn_blocks_properties_df['is_right_column'] = 0 # Whether current column is the right column on the page
            leftcolumn_blocks_properties_df['index_in_column'] = list(range(len(leftcolumn_blocks_properties_df))) # Index the blocks within the column
            # Right-column blocks sorting and indexing
            rightcolumn_blocks_properties_df = mainbody_wizref_properties_df.query("upperleft_x_coordinate > @central_axis_x_coordinate")
            rightcolumn_blocks_properties_df['is_right_column'] = 1 # Whether current column is the right column on the page
            rightcolumn_blocks_properties_df['index_in_column'] = list(range(len(rightcolumn_blocks_properties_df))) # Index the blocks within the column
            # Reconcatenate the two sorted property matrices by row, and save back to variable 'mainbody_wizref_properties_df'.
            mainbody_wizref_properties_df = pd.concat([
                leftcolumn_blocks_properties_df.sort_values(by="upperleft_y_coordinate"),
                rightcolumn_blocks_properties_df.sort_values(by="upperleft_y_coordinate")
            ], axis=0)

            ## 1.4(a) [Output: Preparation] Match block property matrix with block plain text table by column "index_in_page", and save to variable "mainbody_wizref_fulltext_df"
            blocks_list_df = pd.DataFrame({
                'index_in_page': [block[5] for block in blocks_list],
                'contents': [block[4] for block in blocks_list]
            })
            mainbody_wizref_fulltext_df = pd.merge(mainbody_wizref_properties_df, blocks_list_df, on='index_in_page', how="left")

            ## 1.4(b) [Output: Operation] Create or concatenate to variable 'allpages_mainbody_wizref_properties_df'
            if is_firstpage:
                allpages_mainbody_wizref_properties_df = mainbody_wizref_fulltext_df.copy()
                is_firstpage = False # All the pages afterwards would not be the first page
            else:
                allpages_mainbody_wizref_properties_df = pd.concat([allpages_mainbody_wizref_properties_df, mainbody_wizref_fulltext_df], axis=0)

        ### 2. Determine sectional affiliation of each block in the PDF file
        # .cumsum() function allows us to assign a unique index for blocks within each section, as it creates a list where each item is the sum of all previous elements (and boolean values are mapped to 0-1 dichotomy)
        allpages_mainbody_wizref_properties_df['section_num'] = allpages_mainbody_wizref_properties_df['is_sec_title'].cumsum()
        # We mask the section_num == 0 segment to exclude introduction section (in most occassions without a title). Then we group the rows by the section they belong to and assign the value of "contents" in the first row (i.e., section title) to the section_name property of all blocks in the same segment (using .group_by("section_num").transform("first") function).
        allpages_mainbody_wizref_properties_df['section_name'] = allpages_mainbody_wizref_properties_df.mask(allpages_mainbody_wizref_properties_df['section_num']==0).groupby('section_num')['contents'].transform('first') 

        ### 3. Extract 'Government Policies and Programs' Section from the PDF file, and deal with paragraphs spanning across two column
        # Retain only blocks belonging to 'Government Policies and Programs' Section
        govpolicy_blocks_df = allpages_mainbody_wizref_properties_df[allpages_mainbody_wizref_properties_df['section_name'].str.contains("Government Policies")]
        # Join text scattering across blocks and prettify before output
        govpolicy = str()
        is_firstblock = True
        for _, block in govpolicy_blocks_df.query("not is_sec_title").iterrows(): # Remove title before construct an iterator
            current_nolinebreak = block['contents'].replace('\n', '') # Remove all line breaks "\n" in the curent block
            if is_firstblock: # Create variable 'govpolicy' for saving the output
                govpolicy = current_nolinebreak
                is_firstblock = False
            else:
                # IMPORTANT: If a block is not first-line-indented, and it's not a subtitle (note we have excluded title from the loop), then it should be a truncated block whose contents should belong to the previous block.
                if not block['is_firstline_indented'] and not block['is_sec_subtitle']:
                    govpolicy = govpolicy + ' ' + current_nolinebreak # Join the plain text in two blocks with a blank
                else: # Otherwise, the block must be a new paragraph or a subtitle
                    govpolicy = govpolicy + '\n' + current_nolinebreak # Join the plain text in two blocks with a line break
        
        ### 4. Save the prettified plain text to TXT file
        txt_name = pdf_names[i].split('.')[0] + "_policy.txt" # Extract the suffix-free name of the PDF file and replace it with "_policy.txt" suffix
        txt_path = os.path.join(txt_dir, txt_name)
        with open(txt_path, 'w', encoding='utf-8') as file:
            file.write(govpolicy) # Output prettified text

        # --------------End the timer--------------
        end_time = time.perf_counter()
        end_time_legible = str(datetime.now())
        execution_time = end_time - start_time # Calculate execution interval
        message_end = f"Successfully saved to {txt_path}. Execution time {execution_time:.2f} seconds"
        print(message_end)
        log.write(end_time_legible + '\n')
        log.write(message_end + '\n\n') # Add two line breaks
        # -----------------------------------------

PDF loaded. Processing 1/159 [Argentina in 2005 from: ../../data/original_report/usgs_southamerica/Argentina_2005.pdf]
Successfully saved to ../../data/policy_section_txt/usgs_southamerica/Argentina_2005_policy.txt. Execution time 0.35 seconds
PDF loaded. Processing 2/159 [Argentina in 2006 from: ../../data/original_report/usgs_southamerica/Argentina_2006.pdf]
Successfully saved to ../../data/policy_section_txt/usgs_southamerica/Argentina_2006_policy.txt. Execution time 0.18 seconds
PDF loaded. Processing 3/159 [Argentina in 2007 from: ../../data/original_report/usgs_southamerica/Argentina_2007.pdf]
Successfully saved to ../../data/policy_section_txt/usgs_southamerica/Argentina_2007_policy.txt. Execution time 0.26 seconds
PDF loaded. Processing 4/159 [Argentina in 2008 from: ../../data/original_report/usgs_southamerica/Argentina_2008.pdf]
Successfully saved to ../../data/policy_section_txt/usgs_southamerica/Argentina_2008_policy.txt. Execution time 0.18 seconds
PDF loaded. Processing 5

In [ ]:
i

81